In [75]:
import fastf1
import pandas as pd
import os

fastf1.Cache.enable_cache('/Users/alihadidi/Desktop/Thesis/F1Proj/data/cache/')


In [76]:
# Test one session before writing any loop
test_session = fastf1.get_session(2023, 1, 'Q')
test_session.load(laps=False, telemetry=False, weather=False, messages=False)

print("Columns available:")
print(test_session.results.columns.tolist())
print()
print(test_session.results[['Abbreviation', 'FullName', 'TeamName', 'Position', 'Q1', 'Q2', 'Q3']])


core           INFO 	Loading data for Bahrain Grand Prix - Qualifying [v3.8.2]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '16', '55', '14', '63', '44', '18', '31', '27', '4', '77', '24', '22', '23', '2', '20', '81', '21', '10']


Columns available:
['DriverNumber', 'BroadcastName', 'Abbreviation', 'DriverId', 'TeamName', 'TeamColor', 'TeamId', 'FirstName', 'LastName', 'FullName', 'HeadshotUrl', 'CountryCode', 'Position', 'ClassifiedPosition', 'GridPosition', 'Q1', 'Q2', 'Q3', 'Time', 'Status', 'Points', 'Laps']

   Abbreviation         FullName         TeamName  Position  \
1           VER   Max Verstappen  Red Bull Racing       1.0   
11          PER     Sergio Perez  Red Bull Racing       2.0   
16          LEC  Charles Leclerc          Ferrari       3.0   
55          SAI     Carlos Sainz          Ferrari       4.0   
14          ALO  Fernando Alonso     Aston Martin       5.0   
63          RUS   George Russell         Mercedes       6.0   
44          HAM   Lewis Hamilton         Mercedes       7.0   
18          STR     Lance Stroll     Aston Martin       8.0   
31          OCO     Esteban Ocon           Alpine       9.0   
27          HUL  Nico Hulkenberg     Haas F1 Team      10.0   
4           NOR    

In [77]:
def extract_qualifying(session):
    df = session.results[['Abbreviation', 'FullName', 'TeamName', 'Position', 'Q1', 'Q2', 'Q3']].copy()
    df = df.reset_index(drop=True)
    df['Q1_s'] = df['Q1'].dt.total_seconds()
    df['Q2_s'] = df['Q2'].dt.total_seconds()
    df['Q3_s'] = df['Q3'].dt.total_seconds()
    df = df.drop(columns=['Q1', 'Q2', 'Q3'])
    df = df.rename(columns={'Position': 'grid_position'})
    return df


def extract_race(session):
    df = session.results[['Abbreviation', 'Position', 'Status']].copy()
    df = df.rename(columns={'Position': 'finish_position'})
    return df


In [84]:
all_records = []

for year in range(2025, 2026):
    schedule = fastf1.get_event_schedule(year, include_testing=False)
    for _, event in schedule.iterrows():
        if event['RoundNumber'] == 0:
            continue
        if 'sprint' in event['EventFormat'].lower():
            print(f"Skipping {year} R{event['RoundNumber']} {event['EventName']} — sprint weekend")
            continue

        try:
            round_num = int(event['RoundNumber'])

            q_session = fastf1.get_session(year, round_num, 'Q')
            q_session.load(laps=False, telemetry=False, weather=False, messages=False)

            r_session = fastf1.get_session(year, round_num, 'R')
            r_session.load(laps=False, telemetry=False, weather=False, messages=False)

            q_data = extract_qualifying(q_session)
            r_data = extract_race(r_session)

            merged = q_data.merge(r_data, on='Abbreviation', how='left')
            merged['season'] = year
            merged['round'] = round_num
            merged['event_name'] = event['EventName']
            merged['circuit'] = event['Location']

            all_records.append(merged)
            print(f"Collected {year} R{round_num} {event['EventName']}")
        
        
            time.sleep(8)   # ~7-8 sessions per race × 8s = stay under 500/hr


        except Exception as e:
            print(f"Failed {year} R{int(event['RoundNumber'])} {event['EventName']}: {e}")


core           INFO 	Loading data for Australian Grand Prix - Qualifying [v3.8.2]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core           INFO 	Finished loading data for 20 drivers: ['4', '81', '1', '63', '22', '23', '16', '44', '10', '55', '6', '14', '18', '7', '5', '12', '27', '30', '31', '87']
core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.2]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core           INFO 	Finished loading data for 20 drivers: ['4', '1', '63', '12', '23', '18', '27', '16', '81', '44', '10', '22', '31', '87', '30', '5', '14', '55', '7', '6']


Collected 2025 R1 Australian Grand Prix


core           INFO 	Loading data for Japanese Grand Prix - Qualifying [v3.8.2]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


Skipping 2025 R2 Chinese Grand Prix — sprint weekend


core           INFO 	Finished loading data for 20 drivers: ['1', '4', '81', '16', '63', '12', '6', '44', '23', '87', '10', '55', '14', '30', '22', '27', '5', '31', '7', '18']
core           INFO 	Loading data for Japanese Grand Prix - Race [v3.8.2]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
core           INFO 	Finished loading data for 20 drivers: ['1', '4', '81', '16', '63', '12', '44', '6', '23', '87', '14', '22', '10', '55', '7', '27', '30', '31', '5', '18']


Collected 2025 R3 Japanese Grand Prix


... (38 outputs left out)

core           INFO 	Loading data for Abu Dhabi Grand Prix - Qualifying [v3.8.2]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


Skipping 2025 R23 Qatar Grand Prix — sprint weekend


core           INFO 	Finished loading data for 20 drivers: ['1', '4', '81', '63', '16', '14', '5', '31', '6', '22', '87', '55', '30', '12', '18', '44', '23', '27', '10', '43']
core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.8.2]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
core           INFO 	Finished loading data for 20 drivers: ['1', '81', '4', '16', '63', '14', '31', '44', '27', '18', '5', '87', '55', '22', '12', '23', '6', '30', '10', '43']


Collected 2025 R24 Abu Dhabi Grand Prix


In [85]:
output_path = '/Users/alihadidi/Desktop/Thesis/F1Proj/data/processed/raw_merged.csv'
os.makedirs(os.path.dirname(output_path), exist_ok=True)

columns = ['season', 'round', 'event_name', 'circuit',
           'Abbreviation', 'FullName', 'TeamName',
           'grid_position', 'Q1_s', 'Q2_s', 'Q3_s',
           'finish_position', 'Status']


df_all = pd.concat(all_records, ignore_index=True)[columns]
df_all.to_csv(output_path, mode='a', header=False, index=False, encoding='utf-8-sig')
print(f"Appended {len(df_all)} rows")

# df_all = pd.concat(all_records, ignore_index=True)[columns]
# df_all.to_csv(output_path, index=False, encoding='utf-8-sig')
# print(f"Saved {len(df_all)} rows to {output_path}")


Appended 360 rows


In [86]:
df_check = pd.read_csv(output_path, encoding='utf-8-sig')
print(f"Shape: {df_check.shape}")
print(f"\nColumns: {df_check.columns.tolist()}")
print(f"\nMissing values:\n{df_check.isna().sum()}")
print(f"\nFirst 20 rows:")
df_check.head(20)


Shape: (3377, 13)

Columns: ['season', 'round', 'event_name', 'circuit', 'Abbreviation', 'FullName', 'TeamName', 'grid_position', 'Q1_s', 'Q2_s', 'Q3_s', 'finish_position', 'Status']

Missing values:
season                0
round                 0
event_name            0
circuit               0
Abbreviation          0
FullName              0
TeamName              0
grid_position         3
Q1_s                 46
Q2_s                897
Q3_s               1725
finish_position       4
Status                1
dtype: int64

First 20 rows:


,season,round,event_name,circuit,Abbreviation,FullName,TeamName,grid_position,Q1_s,Q2_s,Q3_s,finish_position,Status
0,2017,1,Australian Grand Prix,Melbourne,HAM,Lewis Hamilton,Mercedes,1.0,84.191,83.251,82.188,2.0,Finished
1,2017,1,Australian Grand Prix,Melbourne,VET,Sebastian Vettel,Ferrari,2.0,85.210,83.401,82.456,1.0,Finished
2,2017,1,Australian Grand Prix,Melbourne,BOT,Valtteri Bottas,Mercedes,3.0,84.514,83.215,82.481,3.0,Finished
3,2017,1,Australian Grand Prix,Melbourne,RAI,Kimi Räikkönen,Ferrari,4.0,84.352,83.376,83.033,4.0,Finished
4,2017,1,Australian Grand Prix,Melbourne,VER,Max Verstappen,Red Bull,5.0,84.482,84.092,83.485,5.0,Finished
5,2017,1,Australian Grand Prix,Melbourne,GRO,Romain Grosjean,Haas F1 Team,6.0,85.419,84.718,84.074,20.0,Water leak
6,2017,1,Australian Grand Prix,Melbourne,MAS,Felipe Massa,Williams,7.0,85.099,84.597,84.443,6.0,Finished
7,2017,1,Australian Grand Prix,Melbourne,SAI,Carlos Sainz,Toro Rosso,8.0,85.542,84.997,84.487,8.0,+1 Lap
8,2017,1,Australian Grand Prix,Melbourne,KVY,Daniil Kvyat,Toro Rosso,9.0,85.970,84.864,84.512,9.0,+1 Lap
9,2017,1,Australian Grand Prix,Melbourne,RIC,Daniel Ricciardo,Red Bull,10.0,85.383,83.989,NaN,17.0,Power Unit


In [87]:
df_check = pd.read_csv('/Users/alihadidi/Desktop/Thesis/F1Proj/data/processed/raw_merged.csv')
print(df_check.groupby('season')['round'].max())


season
2017    20
2018    21
2019    21
2020    17
2021    22
2022    22
2023    22
2024    24
2025    24
Name: round, dtype: int64
